<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 13 · HJM and LMM Perspectives

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch13_hjm_lmm_perspectives.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Whole-curve models treat forward rates, not a single short rate, as the modeling object. In this notebook you reproduce the chapter's one-factor HJM drift proxy from the frozen forward-rate and volatility inputs, inspect simulated forward-curve paths under a common tenor shock, and apply one lognormal LMM-style step to the market forwards. You learn why no-arbitrage ties the forward-rate drift to the volatility structure once volatility is chosen, which tenor, measure, and correlation controls must be stated before a result is auditable, and what the teaching sketch deliberately leaves out compared with a production HJM or LMM implementation.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script loads the frozen forward-curve snapshot and volatility assumptions, computes the one-factor HJM drift proxy at each tenor, and prints the table. It then simulates a few forward-rate paths with a common tenor shock, prints the first rows, and finally applies one lognormal LMM-style step to the market forwards.

In [ ]:
SCRIPT = ROOT / "code" / "ch13_hjm_lmm_perspectives.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Read the tables with the modeling boundary in view.

- Confirm the inputs are decimals: forward rates run from 3.4% to 4.1% and volatilities from 0.006 to 0.010, not basis points.
- Check the drift proxy: it grows with maturity, from 0.000018 at the 0.5Y node to 0.000435 at 5Y, because it multiplies local volatility by the tenor-scaled integrated volatility.
- Read the simulated paths: the common one-factor shock moves all tenors together, and the drift adds a small deterministic tilt on top.
- Note the evolved LMM-style forwards move by tiny amounts, for example 0.033949 at the 0.5Y node, because the fixed shock vector and quarter-year step keep the move illustrative.
- Remember the drift proxy is a teaching sketch, not a measure-consistent calibrated HJM drift, and must not be used for valuation.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch13_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: HJM drift consistency on a discrete grid

In one-factor HJM, no-arbitrage forces the integrated forward drift to satisfy a precise identity once volatility is chosen: the drift integral up to T equals one half of the squared volatility integral up to T. This mini-lab checks that identity on the chapter's discrete tenor grid with piecewise-constant volatility, using the same step rule as the script's drift proxy. The identity is exact in continuous time, so the printed residual quantifies the discretization gap.

In [ ]:
import numpy as np

maturity = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 5.0])
vol = np.array([0.006, 0.007, 0.008, 0.008, 0.009, 0.010])
dtau = np.diff(maturity, prepend=0.0)  # tenor interval widths
cum = np.cumsum(vol * dtau)  # integrated volatility
drift = vol * cum  # discrete HJM drift proxy
lhs = np.cumsum(drift * dtau)  # integrated drift, step rule
rhs = 0.5 * cum ** 2  # no-arbitrage identity target
for m, a, b in zip(maturity, lhs, rhs):
    print(f"T={m:4.1f}  drift integral={a:.6f}  "
          f"0.5*S^2={b:.6f}  gap={a - b:.2e}")

### What to observe

The step-rule drift integral overshoots the 0.5 * S^2 target at every tenor, for example 0.000132 versus 0.000105 at the 2Y node. The gap equals one half of the cumulated squared volatility increments, (vol * dtau)^2, and is therefore second order in the tenor width: refining the grid closes it quickly. The identity itself is what stops drift and volatility from being chosen independently.

## Exercises

1. **Recompute the drift with flat volatility.** Set all volatilities to 0.008 and rebuild the drift proxy column. *Hint:* With flat volatility the integrated volatility is 0.008 * T and the drift becomes 0.008^2 * T at each node.
2. **Test the identity on a finer grid.** Insert intermediate tenor nodes and rerun the consistency check. *Hint:* The gap is one half of the cumulated (vol * dtau)^2 terms, so halving the interval widths cuts the gap roughly by a factor of four.
3. **Compare the two evolution styles.** Explain why the HJM sketch evolves forwards additively while the LMM-style step is multiplicative. *Hint:* HJM works with instantaneous forwards that may be negative; the lognormal LMM step keeps market tenor forwards positive by construction.
4. **State the missing controls.** List the fields a production model note must add beyond drift and volatility before the result is auditable. *Hint:* Tenor grid, discounting and projection curves, correlation across forwards, and the valuation measure.
5. **Attribute the largest moves.** In the evolved column, identify the largest up and down moves and explain why the 5Y node barely moves. *Hint:* The move scales with vol * sqrt(dt) * shock; the 5Y shock is zero, so only the small -0.5 * vol^2 * dt term remains.

For more detail, see Chapter 13 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">